In [0]:
import configparser
config=configparser.ConfigParser()
config.read('/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/config2.ini')

Insertion of new files

In [0]:
 %run ./ingestion_utils

In [0]:
new_files = get_new_files(config["VOLUME"]["volume_path"],
config['TABLES']['processed_file_table']).collect()

In [0]:

for file_data in new_files:
    print(file_data['file_path'])
    table_name=get_table_name(config,file_data['source_name'])
    print(table_name)
    try :

      df =load_incremental_file(file_data['file_path'],table_name)
      write_bronze(df,table_name)
      update_file_tracker(file_data)
      log_ingestion(
        file_data["source_name"],
            table_name,
            file_data["file_name"],
            file_data["file_path"],
            "processed",
            df.count(),
            "no error"
      )
    except Exception as e:
      log_ingestion(
            file_data["source_name"],
            table_name,
            file_data["file_name"],
            file_data["file_path"],
            "failed",
            0,
            str(e)
        )
      print(f"Error processing {file_data['file_name']}: {e}")
      
